# Discover - gcode

Cet atelier présente le *GCode*, le langage qui pilote les machines-outils à commande numérique (fraiseuses, imprimantes 3D, traceurs, etc.). Le support de travail est un traceur (*pen-plotter*) simulé dans ce notebook. Les notions abordées (déplacements, absolu/relatif, arcs de cercle, réglages persistants) se transposent aux autres machines-outil. L'atelier se déroule en trois temps : **lire** du GCode, en **écrire** à la main, puis en **générer** par un petit programme.

<details>
<summary>Détails</summary>

**titre:** Introduction au GCode, instructions et tracés machine  
**auteur:** Antoine Gros  
**création:** 12/10/26  
**dernière édition:** 3/11/26  
**mots-clefs:** manufacturing, pen-plotting, 3d printing, gcode  
**durée**: ~= 1h

**Abstract**  
Cet atelier présente le *GCode*, le langage qui pilote les machines-outils à commande numérique (fraiseuses, imprimantes 3D, traceurs, etc.). Le support de travail est un traceur (*pen-plotter*) simulé dans ce notebook. Les notions abordées (déplacements, absolu/relatif, arcs de cercle, réglages persistants) se transposent aux autres machines-outil. L'atelier se déroule en trois temps : **lire** du GCode, en **écrire** à la main, puis en **générer** par un petit programme.

**Prérequis**
+ concepts et vocabulaire de la Conception-Fabrication Assitée par Ordinateur (CFAO)
+ CAO, niveau débutant

**Objectifs**
+ *comprendre* le fonctionnement de la commande numérique : une machine, une mémoire, des instructions ;
+ *lire* un programme G-code ligne par ligne et *prévoir* ce que la machine va faire, avant même de la lancer ;
+ *écrire* un programme complet, à la main ou via un autre programme ; 

</details>

### Table des matières

- **Prologue**
    - Comment fonctionne un notebook Jupyter ?
    - Démarrer la boîte à outils
    - Ressources et dictionnaire des commandes
- **I. Introduction**
    - I.1. GCode pour machines-outils à commande numérique
    - I.2. Anatomie d'un programme
    - I.3. Une machine à états
    - I.4. Une recette et des conventions
- **II. (Exercices) Lire et écrire du GCode**
    - II.1. Quatre programmes à lire
    - II.2. Deux quadrilatères
    - II.3. Débogage
    - II.4. Absolu et relatif, `G90` et `G91`
    - II.5. Suivez le plan
- **III. Automatiser**
- **Appendices** : A. Aide-mémoire ; B. Débogage

## Prologue

### Comment fonctionne un *notebook Jupyter* ?

Ce document est un **notebook** (cahier): un environnement de programmation intéractif permettant à la fois d'afficher du texte et d'exécuter des programmes. Un notebook est structuré en une suite de **cellules** contenant du texte ou du code. Une cellule de code s'execute avec ▶ ou _Maj + Entrée_.


| Type de cellule | À quoi ça ressemble | Que faire ? |
|---|---|---|
| **Texte** | du texte mis en forme, pas de numéro à gauche | Lire. |
| **Code** | un cadre gris avec `[ ]` à sa gauche | Sélectionner puis appuyer sur *Maj + Entrée* (⇧ + ↵) pour l'*exécuter*. Le résultat s'affiche juste en dessous, et le `[ ]` se remplit d'un numéro. |


<div style="border-left:6px solid #4a6fa5;background:rgba(74,111,165,.10);padding:4px 18px;border-radius:6px;margin:8px 0">

**ℹ️ Notes**

1. **Exécutez les cellules de haut en bas, dans l'ordre.** Une cellule peut dépendre de celles qui la précèdent.
2. Dans une cellule de code, vous pouvez **modifier le code** puis le ré-exécuter. C'est fait pour ça.
3. Si vous êtes perdu : menu **Noyau → Redémarrer** (eng. *Kernel → Restart*), puis **ré-exécutez les cellules depuis le début**
4. Si une cellule est trop longue, vous pouvez la **rétracter** en double-cliquant à gauche de celle-ci ou en allant dans le menu déroulant de la cellule.
5. Les menus **🔎 Indice** et **✅ Solution** se déplient d'un clic. 
6. Dans les **cellules de code**, certaines lignes sont commentées par un `#`, elles ne s'executeront pas. Vous pouvez commenter/décommenter une ligne en enlevant/ajoutant le `#`.

</div>


### Démarrer la boîte à outils
Les exercices de ce *notebook* dépendent d'un ensemble de *fonctions* écrites en *Python*.  
Elles permettent de visualiser et contrôler des instructions simples qui seraient envoyés vers un *plotter*. Les deux fonctions mises à disposition sont `show()` et `check()`: 

| Fonction | Ce qu'elle fait |
|---|---|
| `show(gcode)` | **dessine** ce que la machine ferait avec votre programme, et signale les erreurs |
| `show(gcode, target="ex4")` | idem, en **superposant l'objectif** de l'exercice (trait vert épais) |
| `show_target("ex7")` | montre l'**objectif** d'un exercice, seul |
| `check(gcode, "ex4")` | **compare** votre dessin à l'objectif et explique ce qui diffère |

<div style="border-left:6px solid #8e44ad;background:rgba(142,68,173,.11);padding:4px 18px;border-radius:6px;margin:8px 0">

**🧪 Exp.0. Simulations**

Au cours de cet atelier, rendez-vous ponctuellement sur https://ncviewer.com/, et simulez le parcours de l'outil. Des visualisations fréquentes de son parcours peuvent vous aider à réaliser plus facilement certains exercices. Vous pouvez aussi travailler dans le simulateur et vérifier vos réponses avec ce notebook.

</div>

Les trois cellules suivantes installent les modules nécéssaires, les importent dans l'environnement de programmation et déclarent les fonctions. Ces cellules sont rétractées, leur lecture ne rentre pas dans le cadre du cours.

In [ ]:
# install modules
# !pip install numpy matplotlib
# !uv add numpy matplotlib ipykernel

In [ ]:
# imports
import html
import math
import re
from types import SimpleNamespace as NS

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from matplotlib.ticker import MultipleLocator
from IPython.display import HTML, display

In [ ]:
# fonctions
# ──────────────────────────────────────────────────────────────────────────────
#  Mini-simulateur de traceur. Trois fonctions sont mises à disposition :
#    show(programme, target=None, labels="auto", grid=None)  dessine ce que ferait la machine
#    show_target(exercice, labels=True, grid=None)           montre l'objectif d'un exercice
#    check(programme, exercice)                              compare votre dessin à l'objectif
#  Convention : origine en bas à gauche d'une feuille A4 paysage (297 × 210 mm), X → , Y ↑ , Z > 0 = stylo levé.
# ──────────────────────────────────────────────────────────────────────────────
PAPER_W, PAPER_H, PEN_UP_Z, EPS = 297.0, 210.0, 5.0, 1e-9
RAPID, FEED0 = 6000.0, 1000.0                      # mm/min : vitesse supposée de G0, et de G1 si F n'est jamais défini
INK, GREY, GOOD, ARC, PAPER = "#14365d", "#8b9099", "#2e8b57", "#8e44ad", "#fbfaf4"
KNOWN_G = (0, 1, 2, 3, 17, 20, 21, 90, 91, 94)
WORD = re.compile(r"([A-Za-z])\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+))")

CSS = """<style>
.gl{font:14px/1.5 system-ui,-apple-system,"Segoe UI",sans-serif;margin:6px 0;padding:10px 14px;border-radius:8px;border:1px solid}
.gl-ok{background:rgba(46,139,87,.13);border-color:rgba(46,139,87,.6)}
.gl-warn{background:rgba(194,94,0,.13);border-color:rgba(194,94,0,.6)}
.gl-bad{background:rgba(192,57,43,.13);border-color:rgba(192,57,43,.6)}
.gl-info{background:rgba(20,54,93,.08);border-color:rgba(20,54,93,.35)}
.gl ul{margin:4px 0 0 18px;padding:0}
.gl code{font-family:ui-monospace,Menlo,Consolas,monospace;background:rgba(128,128,128,.18);padding:1px 5px;border-radius:4px}
</style>"""


def _esc(s):
    return html.escape(str(s), quote=False)


def _box(kind, body):
    display(HTML(f'{CSS}<div class="gl gl-{kind}">{body}</div>'))


def _dur(t):
    return "moins d'1 s" if t < 1 else f"{t:.0f} s" if t < 60 else f"{t // 60:.0f} min {t % 60:02.0f} s"


def _fmt(v):
    """40 -> '40', 14.1421 -> '14.1'"""
    s = f"{v:.1f}".rstrip("0").rstrip(".")
    return "0" if s in ("", "-0") else s


# ── 1. L'interpréteur : lit le programme ligne par ligne et simule la machine ─────────────
def run(program):
    text = "\n".join(map(str, program)) if isinstance(program, (list, tuple)) else str(program)
    lines = text.replace("\r", "").lstrip("\n").split("\n")      # la 1re ligne d'instruction est la ligne n°1
    st = NS(x=0.0, y=0.0, z=PEN_UP_Z, rel=False, k=1.0, feed=None, mode=None, end=False)   # la « mémoire » de la machine
    segs, events, errors, warns, once = [], [], [], [], set()

    def warn(n, key, msg, only_once=False):
        if not (only_once and key in once):
            once.add(key)
            warns.append((n, key, msg))

    def move(n, m, P):
        k = st.k
        tgt = lambda a, cur: cur if a not in P else (cur + P[a] * k if st.rel else P[a] * k)
        nx, ny, nz = tgt("X", st.x), tgt("Y", st.y), tgt("Z", st.z)
        down0, down1 = st.z <= EPS, nz <= EPS
        arc = None
        if m in (0, 1):
            pts = np.array([[st.x, st.y], [nx, ny]])
            length = math.hypot(nx - st.x, ny - st.y)
        else:
            if "X" not in P and "Y" not in P:
                return errors.append((n, "G2/G3 : indiquez le point d'arrivée avec X et/ou Y."))
            if "I" not in P and "J" not in P:
                return errors.append((n, "G2/G3 : indiquez le centre de l'arc avec I et J "
                                         "(distances du point de DÉPART jusqu'au centre)."))
            cx, cy = st.x + P.get("I", 0.0) * k, st.y + P.get("J", 0.0) * k
            r0, r1 = math.hypot(st.x - cx, st.y - cy), math.hypot(nx - cx, ny - cy)
            if r0 < EPS:
                return errors.append((n, "Rayon nul : I et J valent 0, le centre serait au point de départ."))
            if abs(r0 - r1) > 0.2:
                warn(n, "arc", f"Arc incohérent : le centre est à {_fmt(r0)} mm du départ mais à {_fmt(r1)} mm de "
                               "l'arrivée. Vérifiez I, J et le point d'arrivée.")
            ccw = m == 3
            a0, a1 = math.atan2(st.y - cy, st.x - cx), math.atan2(ny - cy, nx - cx)
            sweep = ((a1 - a0) if ccw else (a0 - a1)) % (2 * math.pi)
            if sweep < 1e-7 or abs(sweep - 2 * math.pi) < 1e-7:          # départ = arrivée : cercle complet
                sweep = 2 * math.pi
            t = np.linspace(0.0, 1.0, int(min(3000, max(16, math.ceil(sweep * max(r0, r1))))) + 1)
            ang, rad = a0 + (sweep if ccw else -sweep) * t, r0 + (r1 - r0) * t
            pts = np.column_stack([cx + rad * np.cos(ang), cy + rad * np.sin(ang)])
            pts[0], pts[-1] = (st.x, st.y), (nx, ny)
            length = float(np.hypot(*np.diff(pts, axis=0).T).sum())
            arc = NS(center=(cx, cy))
        if m in (2, 3) or length > EPS:
            drawn = down0 and down1
            if abs(nz - st.z) > EPS:
                warn(n, "xyz", "X/Y et Z changent sur la même ligne : séparez le mouvement du stylo (Z) du mouvement dans le plan.")
            if m == 0 and drawn:
                warn(n, "g0pen", "G0 (rapide) avec le stylo posé : il traîne sur le papier. Convention : G0 stylo levé, G1 pour tracer.")
            if m != 0 and st.feed is None:
                warn(n, "nofeed", "Aucune vitesse F n'a été définie avant ce tracé : le simulateur suppose 1000 mm/min.", True)
            if ((pts[:, 0] < -1e-6) | (pts[:, 0] > PAPER_W + 1e-6) | (pts[:, 1] < -1e-6) | (pts[:, 1] > PAPER_H + 1e-6)).any():
                warn(n, "area", f"{'Le tracé' if drawn else 'Le déplacement'} sort de la feuille A4 "
                                f"({_fmt(PAPER_W)} × {_fmt(PAPER_H)} mm).")
            segs.append(NS(n=n, pts=pts, drawn=drawn, length=length, rapid=m == 0, arc=arc,
                           feed=st.feed if m else None))
        if down0 != down1:
            events.append(NS(down=down1, x=nx, y=ny))
        st.x, st.y, st.z = nx, ny, nz

    def step(n, code):
        fail = lambda msg: errors.append((n, msg))
        if st.end:
            return warn(n, "after", f"Les lignes après M2 (à partir de la ligne {n}) ne sont jamais exécutées.", True)
        if "_" in code or "?" in code or "..." in code:
            return fail("Il reste un trou (___ ou ...) à remplacer par un nombre ou par vos propres lignes de G-code.")
        if re.search(r"\d,\d", code):
            return fail("Utilisez le POINT et non la virgule pour les décimales (12.5 et non 12,5).")
        rest = WORD.sub(" ", code).strip()
        if rest:
            return fail(f"Je ne comprends pas « {rest} » : chaque lettre doit être suivie d'un nombre (par exemple X50).")
        words = [(m.group(1).upper(), float(m.group(2))) for m in WORD.finditer(code)]
        G = [v for l, v in words if l == "G"]
        M = [v for l, v in words if l == "M"]
        P = {l: v for l, v in words if l not in "GM"}
        if "R" in P:
            return fail("Les arcs avec R ne sont pas gérés ici : utilisez I et J (centre de l'arc par rapport au départ).")
        if set(P) - set("XYZIJF"):
            return fail(f"Lettre inconnue : {', '.join(sorted(set(P) - set('XYZIJF')))}. Je connais G, M, X, Y, Z, I, J, F.")
        if any(v != int(v) for v in G + M):
            return fail("Numéro de commande non entier (G1.5 ?) : je ne connais que des numéros entiers.")
        G, M = [int(v) for v in G], [int(v) for v in M]
        if [g for g in G if g not in KNOWN_G]:
            return fail(f"Commande G{[g for g in G if g not in KNOWN_G][0]} inconnue ici. "
                        "Celles qu'on utilise : G0 G1 G2 G3 G17 G20 G21 G90 G91 G94.")
        if [m for m in M if m not in (2, 30)]:
            return fail(f"Commande M{[m for m in M if m not in (2, 30)][0]} inconnue ici. Seule M2 (fin de programme) est gérée.")
        for g in G:                                              # réglages « persistants » de la machine
            if g in (20, 21):
                st.k = 25.4 if g == 20 else 1.0
            elif g in (90, 91):
                st.rel = g == 91
        if "F" in P:
            if P["F"] <= 0:
                return fail("La vitesse F doit être un nombre strictement positif.")
            st.feed = P["F"] * st.k
        mot = [g for g in G if g in (0, 1, 2, 3)]
        if len(mot) > 1:
            return fail("Deux commandes de déplacement sur la même ligne : n'en mettez qu'une (G0, G1, G2 ou G3).")
        if mot:
            st.mode = mot[0]
        if any(a in P for a in "XYZIJ"):
            if st.mode is None:
                return fail("Cette ligne donne des coordonnées mais aucune commande de déplacement n'est active : commencez par G0 ou G1.")
            move(n, st.mode, P)
        if 2 in M or 30 in M:
            st.end = True

    for n, raw in enumerate(lines, start=1):
        code = re.sub(r"\([^)]*\)", " ", raw).split(";", 1)[0].strip()      # on retire les commentaires ; … et ( … )
        if code:
            step(n, code)

    if st.z <= EPS and any(s.drawn for s in segs):
        warns.append((None, "penend", "À la fin du programme, le stylo est encore posé sur le papier : ajoutez G0 Z5 avant M2."))
    stats = NS(drawn=sum(s.length for s in segs if s.drawn), travel=sum(s.length for s in segs if not s.drawn),
               time=sum(s.length / (RAPID if s.rapid else (s.feed or FEED0)) * 60 for s in segs),
               lifts=sum(1 for i, e in enumerate(events) if not e.down and any(f.down for f in events[:i])),
               arcs=sum(1 for s in segs if s.arc is not None and s.drawn))
    return NS(segs=segs, events=events, errors=errors, warns=warns, stats=stats, rel=st.rel, end=st.end)


# ── 2. Le dessin ─────────────────────────────────────────────────────────────────────────
def _dense(pts, step=1.0):
    """Points régulièrement espacés le long d'une polyligne (sert aux flèches et aux comparaisons)."""
    pts = np.asarray(pts, dtype=float)
    out = [pts[:1]]
    for a, b in zip(pts[:-1], pts[1:]):
        t = np.linspace(0, 1, max(1, int(math.ceil(np.hypot(*(b - a)) / step))) + 1)[1:, None]
        out.append(a + (b - a) * t)
    return np.vstack(out)


def _paper(ax, pts, grid):
    """Feuille A4, quadrillage, origine ; cadrage autour des points `pts`."""
    P = np.array(list(pts) + [(0.0, 0.0)], dtype=float)
    (x0, y0), (x1, y1) = P.min(0), P.max(0)
    w, h = max(x1 - x0, 60) * 1.24, max(y1 - y0, 60) * 1.24
    ax.add_patch(Rectangle((0, 0), PAPER_W, PAPER_H, fc=PAPER, ec="#555", lw=1, ls=(0, (4, 3)), zorder=0))
    ax.set_xlim((x0 + x1 - w) / 2, (x0 + x1 + w) / 2)
    ax.set_ylim((y0 + y1 - h) / 2, (y0 + y1 + h) / 2)
    ax.set_aspect("equal", adjustable="box")
    step = grid or next((s for s in (1, 2, 5, 10, 20, 25, 50, 100) if max(w, h) / s <= 12), 200)
    ax.xaxis.set_major_locator(MultipleLocator(step))
    ax.yaxis.set_major_locator(MultipleLocator(step))
    ax.grid(True, color="#d9d9d9", lw=0.7, zorder=1)
    ax.set_axisbelow(True)
    ax.tick_params(labelsize=8)
    ax.set_xlabel("X (mm)  →", fontsize=9)
    ax.set_ylabel("Y (mm)  →", fontsize=9)
    ax.plot([0], [0], "s", color="black", ms=5, zorder=9)
    ax.annotate("origine (0, 0)", (0, 0), xytext=(6, -13), textcoords="offset points", fontsize=8, zorder=9)


def _label(ax, text, xy, color, fs=8, weight="normal"):
    ax.annotate(text, xy, xytext=(5, 6), textcoords="offset points", fontsize=fs, fontweight=weight, color=color, zorder=10,
                bbox=dict(boxstyle="round,pad=0.12", fc="white", ec="none", alpha=0.8))


def show(program, target=None, labels="auto", grid=None):
    """Dessine ce que la machine ferait avec `program`. `target="exN"` superpose l'objectif (en vert)."""
    r = run(program)
    spec = EXERCISES[target] if target else None
    fig, ax = plt.subplots(figsize=(7.2, 5.2), dpi=100)
    pts = [tuple(p) for s in r.segs for p in s.pts[:: max(1, len(s.pts) // 24)]] + [(e.x, e.y) for e in r.events]
    _paper(ax, pts + ([tuple(p) for part in spec["parts"] for p in part["pts"]] if spec else []), grid)
    span = max(np.ptp(ax.get_xlim()), np.ptp(ax.get_ylim()))
    for part in (spec["parts"] if spec else []):
        q = np.array(part["pts"])
        ax.plot(q[:, 0], q[:, 1], color=GOOD, lw=9, alpha=0.28, solid_capstyle="round", solid_joinstyle="round", zorder=2)
    for s in r.segs:
        if not s.drawn:
            ax.plot(s.pts[:, 0], s.pts[:, 1], color=GREY, lw=1.5, ls=(0, (1.5, 2.8)), zorder=4)
            continue
        ax.plot(s.pts[:, 0], s.pts[:, 1], color=INK, lw=2.6, solid_capstyle="round", solid_joinstyle="round", zorder=5)
        d = _dense(s.pts)
        out = (d[:, 0] < -1e-6) | (d[:, 0] > PAPER_W + 1e-6) | (d[:, 1] < -1e-6) | (d[:, 1] > PAPER_H + 1e-6)
        if out.any():                                                  # la partie qui sort de la feuille, en rouge
            ax.plot(np.where(out, d[:, 0], np.nan), np.where(out, d[:, 1], np.nan), color="#c0392b", lw=3.2, zorder=6)
        if s.length >= 0.07 * span and len(d) > 2:                     # flèche au milieu du tracé : sens de parcours
            cum = np.concatenate([[0], np.cumsum(np.hypot(*np.diff(d, axis=0).T))])
            k = min(max(int(np.searchsorted(cum, cum[-1] / 2)), 1), len(d) - 2)
            v = d[k + 1] - d[k - 1]
            if np.hypot(*v) > 0:
                v = v / np.hypot(*v) * 0.012 * span
                ax.annotate("", xy=d[k] + v, xytext=d[k] - v, zorder=7,
                            arrowprops=dict(arrowstyle="-|>", color=INK, lw=1.2, mutation_scale=15, shrinkA=0, shrinkB=0))
    arcs = [s for s in r.segs if s.arc is not None]
    if 0 < len(arcs) <= 6:                                            # le centre de chaque arc, relié aux extrémités
        for s in arcs:
            cx, cy = s.arc.center
            for q in (s.pts[0], s.pts[-1]):
                ax.plot([q[0], cx], [q[1], cy], color=ARC, lw=1, ls=(0, (5, 3)), zorder=3)
            ax.plot(cx, cy, "+", ms=12, mew=2, color=ARC, zorder=8)
            ax.annotate("centre", (cx, cy), xytext=(6, -14), textcoords="offset points", fontsize=8, color=ARC, zorder=9)
    if labels == "auto":
        labels = len(r.segs) <= 24
    if labels:                                                        # n° de la ligne du programme qui a amené le stylo là
        for s in r.segs:
            _label(ax, str(s.n), s.pts[-1], INK if s.drawn else GREY, weight="bold")
    for e in r.events[:30]:
        ax.plot(e.x, e.y, "v" if e.down else "^", ms=8, mec=INK, mfc=INK if e.down else "white", zorder=8)
    h = [Line2D([0], [0], color=INK, lw=2.6, label="trait (stylo posé)"),
         Line2D([0], [0], color=GREY, lw=1.5, ls=(0, (1.5, 2.8)), label="déplacement à vide (stylo levé)")]
    if r.events:
        h += [Line2D([0], [0], marker="v", ls="", mfc=INK, mec=INK, ms=8, label="stylo posé ici"),
              Line2D([0], [0], marker="^", ls="", mfc="white", mec=INK, ms=8, label="stylo levé ici")]
    if arcs:
        h.append(Line2D([0], [0], marker="+", ls="", color=ARC, mew=2, ms=10, label="centre de l'arc"))
    if spec:
        h.append(Line2D([0], [0], color=GOOD, lw=9, alpha=0.28, label="objectif (en vert)"))
    h.append(Rectangle((0, 0), 1, 1, fc=PAPER, ec="#555", ls="--", label="feuille A4"))
    ax.legend(handles=h, loc="upper center", bbox_to_anchor=(0.5, -0.13), ncol=3, fontsize=8, frameon=False)
    fig.tight_layout()
    plt.show()
    s, ok = r.stats, not (r.errors or r.warns)
    body = (f"<b>Tracé</b> : {_fmt(s.drawn)} mm &nbsp;·&nbsp; <b>Déplacements à vide</b> : {_fmt(s.travel)} mm &nbsp;·&nbsp; "
            f"<b>Levées de stylo</b> : {s.lifts} &nbsp;·&nbsp; <b>Durée estimée</b> : {_dur(s.time)}"
            + ("<br><span style='opacity:.8'>✔ Aucune erreur de syntaxe"
               "</span>" if ok else ""))
    _box("info", body)
    if r.errors:
        li = "".join(f"<li><b>Ligne {n}</b> : {_esc(m)}</li>" for n, m in r.errors[:8])
        _box("bad", f"<b>❌ Erreurs</b> (ces lignes sont ignorées par la machine)<ul>{li}</ul>")
    if r.warns:
        li = "".join(f"<li>{'<b>Ligne %d</b> : ' % n if n else ''}{_esc(m)}</li>" for n, _k, m in r.warns[:8])
        _box("warn", f"<b>⚠️ Attention</b><ul>{li}</ul>")


def show_target(ex, labels=True, grid=None):
    """Montre l'objectif d'un exercice (en vert), avec, si `labels`, les coordonnées des sommets."""
    spec = EXERCISES[ex]
    fig, ax = plt.subplots(figsize=(7.2, 5.2), dpi=100)
    _paper(ax, [tuple(p) for part in spec["parts"] for p in part["pts"]], grid)
    for part in spec["parts"]:
        q = np.array(part["pts"])
        ax.plot(q[:, 0], q[:, 1], color=GOOD, lw=3, solid_capstyle="round", solid_joinstyle="round", zorder=5)
    if labels:
        for x, y in spec["vertices"]:
            ax.plot(x, y, "o", color=GOOD, ms=5, zorder=6)
            _label(ax, f"({_fmt(x)}, {_fmt(y)})", (x, y), "#1d5c3a")
    ax.set_title("Objectif : " + spec["title"], fontsize=11)
    fig.tight_layout()
    plt.show()


# ── 3. Les objectifs des exercices : la géométrie est déduite de programmes de référence ──────────────
REFERENCE = {
    "ex2": ("Le carré de 40 mm", "G21 G90|G0 Z5|G0 X10 Y10|G1 Z0 F500|G1 X50 Y10|G1 X50 Y50|G1 X10 Y50|G1 X10 Y10|G0 Z5|M2",
            ["côté du bas", "côté de droite", "côté du haut", "côté de gauche"]),
    "ex2b": ("Le rectangle de 80 × 30 mm", "G21 G90|G0 Z5|G0 X20 Y20|G1 Z0 F500|G1 X100 Y20|G1 X100 Y50|G1 X20 Y50|G1 X20 Y20|G0 Z5|M2",
             ["côté du bas", "côté de droite", "côté du haut", "côté de gauche"]),
    "ex4": ("Un carré et un triangle",
            "G21 G90|G0 Z5|G0 X20 Y20|G1 Z0 F800|G1 X60 Y20|G1 X60 Y60|G1 X20 Y60|G1 X20 Y20|G0 Z5|G0 X80 Y20|G1 Z0|"
            "G1 X110 Y20|G1 X95 Y45|G1 X80 Y20|G0 Z5|M2",
            ["côté du bas du carré", "côté de droite du carré", "côté du haut du carré", "côté de gauche du carré",
             "base du triangle", "côté droit du triangle", "côté gauche du triangle"]),
    "ex5a": ("L'escalier en mode relatif",
             "G21 G90|G0 Z5|G0 X10 Y10|G1 Z0 F1500|G91|" + "G1 X25 Y0|G1 X0 Y15|" * 4 + "G90|G0 Z5|M2",
             [f"{p} {k}" for k in range(1, 5) for p in ("marche", "contremarche")]),
    "ex7": ("Le plan d'une pièce avec sa porte",
            "G21 G90|G0 Z5|G0 X50 Y20|G1 Z0 F1000|G1 X20 Y20|G1 X20 Y90|G1 X120 Y90|G1 X120 Y20|G1 X80 Y20|G0 Z5|"
            "G0 X50 Y20|G1 Z0|G1 X50 Y50|G2 X80 Y20 I0 J-30|G0 Z5|M2",
            ["mur du bas, à gauche de la porte", "mur de gauche", "mur du haut", "mur de droite",
             "mur du bas, à droite de la porte", "le battant de la porte", "l'arc de la porte"]),
    "ex8": ("Les hachures", "G21 G90|G0 Z5|" + "".join(f"G0 X10 Y{10 + 4 * i}|G1 Z0 F1500|G1 X50 Y{10 + 4 * i}|G0 Z5|" for i in range(11)) + "M2",
            [f"hachure n°{i + 1}" for i in range(11)]),
}
EXTRA = {"ex5a": dict(final_absolute=True), "ex7": dict(min_arcs=1, tol=1.0)}
HINTS = {"ex7": {"mur du bas, à gauche de la porte": "le mur du bas est interrompu par l'ouverture de la porte, entre X50 et X80.",
                 "mur du bas, à droite de la porte": "le mur du bas est interrompu par l'ouverture de la porte, entre X50 et X80."}}


def _build_exercises():
    specs = {}
    for ex, (title, ref, names) in REFERENCE.items():
        parts = []
        for s, name in zip([s for s in run(ref.replace("|", "\n")).segs if s.drawn], names):
            pts = [tuple(map(float, q)) for q in (s.pts if s.arc is None else s.pts[:: max(1, len(s.pts) // 60)])]
            parts.append(dict(name=name, pts=pts + [tuple(map(float, s.pts[-1]))] if s.arc else pts))
        verts = []
        for part in parts:
            for q in (part["pts"][0], part["pts"][-1]):
                if q not in verts:
                    verts.append(q)
        specs[ex] = dict(title=title, parts=parts, vertices=verts, hints=HINTS.get(ex, {}), **EXTRA.get(ex, {}))
    return specs


EXERCISES = _build_exercises()


# ── 4. La vérification : compare le dessin de l'étudiant à l'objectif ──────────────────────────────────
def _dist(P, lines):
    """Distance de chaque point de P à l'ensemble des polylignes `lines`."""
    A = np.vstack([np.asarray(l, float)[:-1] for l in lines])
    B = np.vstack([np.asarray(l, float)[1:] for l in lines])
    AB, out = B - A, np.empty(len(P))
    L2 = np.maximum((AB ** 2).sum(1), 1e-12)
    for i in range(0, len(P), 250):
        Q = P[i:i + 250]
        t = np.clip(((Q[:, None] - A[None]) * AB[None]).sum(2) / L2[None], 0, 1)
        out[i:i + 250] = np.linalg.norm(Q[:, None] - (A[None] + t[..., None] * AB[None]), axis=2).min(1)
    return out


def check(program, ex):
    """Compare votre dessin à l'objectif de l'exercice `ex` et explique ce qui diffère."""
    spec, r, bad = EXERCISES[ex], run(program), []
    drawn = [s for s in r.segs if s.drawn]
    if r.errors:
        li = "".join(f"<li>ligne {n} : {_esc(m)}</li>" for n, m in r.errors[:5])
        bad.append(f"Le simulateur ne comprend pas tout le programme. Corrigez d'abord :<ul>{li}</ul>")
    elif any(k == "area" for _n, k, _m in r.warns):
        bad.append("Le dessin sort de la feuille : vérifiez vos nombres (G21 = mm, G20 = pouces !) et, pour un arc, son sens "
                   "(G2 ↻ ou G3 ↺) : un arc dans le mauvais sens fait le grand tour.")
    elif not drawn:
        bad.append("Rien n'est dessiné : le stylo n'est jamais posé sur le papier (<code>G1 Z0</code>).")
    else:
        tol = spec.get("tol", 0.6)
        mine = [s.pts for s in drawn]
        goal = [np.array(p["pts"]) for p in spec["parts"]]
        missing = [(p["name"], float((_dist(_dense(g), mine) <= tol).mean())) for p, g in zip(spec["parts"], goal)]
        missing = [(name, cov) for name, cov in missing if cov < 0.97]
        D = np.vstack([_dense(s.pts) for s in drawn])
        L = np.concatenate([np.full(len(_dense(s.pts)), s.n) for s in drawn])
        far = _dist(D, goal) > tol
        if missing:
            li = "".join(f"<li><b>{_esc(n)}</b> : {'il manque' if c < 0.05 else 'incomplet ou mal placé'}"
                         + (f" — <i>{_esc(spec['hints'][n])}</i>" if n in spec["hints"] else "") + "</li>" for n, c in missing)
            bad.append(f"Il manque une partie du dessin :<ul>{li}</ul>")
        if far.sum() > 1.5:
            lines_ = ", ".join(str(n) for n in sorted(set(L[far]))[:6])
            x, y = D[np.argmax(far)]
            bad.append(f"Du tracé en trop (environ {_fmt(far.sum())} mm), par exemple près de ({_fmt(x)}, {_fmt(y)}) ; "
                       f"produit par la/les ligne(s) {lines_}.")
        if spec.get("min_arcs") and r.stats.arcs < spec["min_arcs"]:
            bad.append("Utilisez une commande d'arc (G2 ou G3) pour la partie courbe.")
        if spec.get("final_absolute") and not bad and r.rel:
            bad.append("Le programme se termine en mode RELATIF. Revenez en mode absolu (<code>G90</code>) dès que les "
                       "déplacements relatifs sont finis, sinon la ligne suivante sera mal interprétée.")
    if bad:
        _box("bad", f"<b>❌ Pas encore : {_esc(spec['title'])}</b>" + "".join(f"<div style='margin-top:6px'>{b}</div>" for b in bad)
             + f"<div style='margin-top:8px;opacity:.85'>Astuce : <code>show(gcode, target=\"{ex}\")</code> superpose l'objectif (en vert) à votre dessin.</div>")
        return
    tips = []
    if not r.end:
        tips.append("terminez le programme par <code>M2</code> (fin de programme)")
    if any(k == "penend" for _n, k, _m in r.warns):
        tips.append("relevez le stylo en fin de programme (<code>G0 Z5</code>)")
    if any(k == "nofeed" for _n, k, _m in r.warns):
        tips.append("définissez une vitesse avec F (par exemple <code>G1 Z0 F1000</code>) avant le premier tracé")
    _box("ok", f"<b>✅ Exercice validé : {_esc(spec['title'])}</b>"
         + "".join(f"<div style='margin-top:6px'>💡 Bonne habitude : {t}.</div>" for t in tips))

### Ressources

Références pour le langage GCode :
+ [RepRap wiki, G-code](https://reprap.org/wiki/G-code) : une description générale du langage et de ses variantes
+ [Marlin G-code index](https://marlinfw.org/meta/gcode/) : le dialecte de nombreuses imprimantes 3D

#### Dictionnaire des commandes et paramètres
Vocabulaire exhaustif des codes utilisés lors de cet atelier, soit les commandes et paramètres de tracé.  
Nous utiliserons uniquement un sous-ensemble des codes existants

| Code | Famille | Signification | Exemple |
|---|---|---|---|
| `G0` | commande | déplacement **rapide**, sans travailler | `G0 X10 Y10` |
| `G1` | commande | déplacement en **ligne droite**, en travaillant, à la vitesse `F` | `G1 X50 Y10` |
| `G2` | commande | **arc** de cercle dans le sens des aiguilles d'une montre ↻ | `G2 X110 Y20 I0 J-90` |
| `G3` | commande | **arc** de cercle dans le sens inverse ↺ | `G3 X20 Y20 I0 J-90` |
| `G21` | commande | changement d'unité en **millimètres** (`G20` : pouces) | `G21` |
| `G90` / `G91` | commande | positions **absolues** / déplacements **relatifs** | `G90` |
| `M2` | commande machine | **fin** du programme | `M2` |
| `X` `Y` `Z` | paramètre | **position** selon les axes | `X50 Y10 Z0` |
| `I` `J` | paramètre | le **centre** de l'arc (par rapport au départ) | `I0 J-90` |
| `F` | paramètre | la **vitesse**, en mm par minute | `F1500` |
| `;` | commentaire | tout ce qui suit est ignoré | `; ma note` |

# I. Introduction

## I.1. GCode pour Machines Outils à Commande Numérique

Le *Gcode* est un langage d'instructions pour Machines Outils à Commande Numérique (MOCN). Il est écrit en *texte brut*, ligne par ligne, avec pour chaque ligne une instruction qui est exécutée en séquence, de haut en bas. Un programme *GCode* décrit uniquement le **trajet que doit suivre l'outil** et les **paramètres de la machine**, il ne contient ni dessin, ni maquette numérique. La géométrie a donc été définie en amont, que ce soit par un dessin, un tracé sur un logiciel de CAO ou de découpe, ou un programme informatique. 

Ce langage est adapté en différents dialectes selon la machine-outil visée et son constructeur [Table 1.]. Le dialecte dépend du *firmware* de la machine qui interprete le *Gcode*[^1]. Les bases (déplacements, unités, positionnement) sont communes, ce sont les commandes auxilliaires qui varient. Dans le cas des imprimantes à Dépôt de Filament Fondu (_Fused Filament Fabrication_), au set d'instruction de base s'ajoutent celles de l'extrusion du filament, la température et les fonctions de ventilation, affichage, alertes sonores. Dans le cas d'une fraiseuse numérique, ce serait la vitesse de rotation de la broche, l'aspiration/soufflerie, voire le changement automatisé d'outil. Certains de ces codes sont des *macros*, des *codes* spécifiques permettant d'invoquer une suite d'instructions pour faciliter la vie de l'utilisateur et pour un usage plus aisé de mouvements machine complexes.

Dans cet atelier, nous travaillerons avec un *traceur* (en. *pen-plotter*). Ces machines utilisent un set limité d'instructions *Gcode* pour piloter le mouvement du stylo dans l'espace. Les exemples donnés sur les autres machines (imprimante 3d, fraiseuse) suivront le dialecte Marlin.


[1]: Le _firmware_, ou micrologiciel, est le logiciel embarqué dans un appareil électronique/informatique, exécuté par son microprocesseur, ayant pour rôle de faire l'interface entre le système électronique, ses fonctions et sa programmation.

> Table 1. Différents dialectes pour différentes machines

| Machine | Fonctions spécifiques | Exemples  |
|---|---|---|
| **Traceur**  | pas de fonction spécifique, le stylo est levé/posé en pilotant l'axe Z | `G0 Z5` |
| **Imprimante 3D** à dépôt de filament fondu (*Fused Filament Fabrication*) | l'extrusion du filament, la température, la ventilation, l'affichage, les alertes sonores | `G1 X50 Y10 E1.2` ; `M104 S210` ; `M106 S255` |
| **Fraiseuse numérique** | la vitesse de rotation de la broche, l'aspiration ou la soufflerie, le changement automatisé d'outil | `M3 S12000` ; `M5` ; `T2 M6` |

### Premier programme

Voic un premier programme en *GCode*. **Lisez-le** et essayez de deviner ce que la machine devrait dessiner. Puis exécutez la cellule

In [ ]:
intro = """
G21 G90          ; unités = millimètres, positions absolues
G0 Z5            ; lever le stylo
G0 X20 Y20       ; aller au point (20, 20)
G1 Z0 F1000      ; poser le stylo
G1 X80 Y50       ; aller au point (80, 50)
G0 Z5            ; lever le stylo
M2               ; fin du programme
"""

show(intro)

<details>
<summary>✅ <i>Solution</i></summary>

Ce programme dessine **un seul trait droit**, du point `(20, 20)` au point `(80, 50)`, soit 60 mm vers la droite et 30 mm vers le haut. Ligne par ligne :

| Ligne | Instruction | Effet |
|---|---|---|
| 1 | `G21 G90` | réglages : millimètres, positions absolues |
| 2 | `G0 Z5` | le stylo se lève |
| 3 | `G0 X20 Y20` | déplacement à vide jusqu'au point (20, 20) : le stylo est levé, **pas de trace** |
| 4 | `G1 Z0 F1000` | le stylo se pose sur le papier ; vitesse de travail 1000 mm/min |
| 5 | `G1 X80 Y50` | trait droit jusqu'à (80, 50) : **c'est la seule ligne qui trace** |
| 6 | `G0 Z5` | le stylo se lève |
| 7 | `M2` | fin du programme |

Les chiffres bleus sur la figure sont ces numéros de ligne : le « 3 » marque l'arrivée du déplacement à vide, le « 5 » celle du trait.

</details>

<div style="border-left:6px solid #2e8b57;background:rgba(46,139,87,.13);padding:4px 18px;border-radius:6px;margin:8px 0">

**💡 Comment lire la figure**

- **Trait bleu plein** : le stylo est *posé*, il dessine.
- **Pointillé gris** : le stylo est *levé*, il se déplace sans laisser de trace.
- **Triangle plein ▼** : le stylo se pose ici. **Triangle vide ▲** : le stylo se lève ici.
- **Numéros** : numéro de la **ligne du programme** qui a amené le stylo à cet endroit.
- **Flèche** : sens du tracé.
- **Carré noir « origine (0, 0) »** : le point de repère de la machine, en bas à gauche de la feuille.
- **Cadre en tirets** : la feuille de papier (A4 format paysage, 297 × 210 mm).
- **Sous la figure** : résumé et, en cas de problème, des messages **rouges** (erreurs) ou **orange** (avertissements).

</div>

## I.2. Anatomie d'un programme

Un programme *GCode* est une suite d'instructions écrites lignes par ligne. Une ligne de *GCode* est une suite de **codes** séparés par des espaces. Un code, c'est une **lettre suivie d'un nombre** (eg. `X50`, `G1`, `F2000`).  

Il y a deux familles de **codes**:
- les codes de **commande** (`G` et `M`) disent *ce que la machine doit faire* : se déplacer, changer d'unité, s'arrêter ;
- les codes de **paramètre** (`X`, `Y`, `Z`, `I`, `J`, `F`) donnent *les valeurs* associées à cette action : eg. où aller, à quelle vitesse.

Exemple: "Va en ligne droite** (G1) **au point** (X10 Y20) **à la vitesse** (F600)"
```text
G1   X10 Y20    F600       ; une commande, trois paramètres, un commentaire
│    └──────┘   └──┘
│    position   vitesse
└─ commande: mouvement linéaire
```

Tout ce qui suit un point-virgule `;` est un **commentaire**, une annotation écrite pour les humains, la machine l'ignore. Commentez vos programmes.

<div style="border-left:6px solid #8e44ad;background:rgba(142,68,173,.11);padding:4px 18px;border-radius:6px;margin:8px 0">

**🧪 Exp.1. Changez un paramètre**

Dans la cellule ci-dessous, la ligne `G1 X80 Y50` donne la destination du trait. **Changez ces deux nombres** et ré-exécutez (Maj + Entrée) à chaque essai. Essayez de prédire le résultat *avant* d'appuyer sur la touche.

Que se passe-t-il quand le point d'arrivée sort de la feuille (par exemple `X350`, ou une valeur négative) ? 

</div>

In [ ]:
intro_exp1 = """
G21 G90          ; unités = millimètres, positions absolues
G0 Z5            ; lever le stylo
G0 X20 Y20       ; aller au point (20, 20)
G1 Z0 F1000      ; poser le stylo
G1 X80 Y50       ; aller au point (80, 50)
G0 Z5            ; lever le stylo
M2               ; fin du programme
"""

show(intro_exp1)

## I.3. Une machine à états

Il est raisonnable de considérer une MOCN comme un [_automate fini_](https://fr.wikipedia.org/wiki/Automate_fini), autrement dit une _machine à états_. C'est le modèle mathématique usuel qui permet de décrire un automate par une succession finie d'état. L’état où l'automate se trouve est appelé _état courant_, le passage à un autre état est dirigé par un événement appelé _transition_. Un automate fini est susceptible d’être (a) dans un nombre fini d’états ou (b) dans un seul état à la fois. 

Dans le cas des MOCN, l'_état courant_ est conservé en **mémoire**. C'est l'exécution d'une instruction en `Gcode` qui provoque la _transition_ de l'_état courant_ à un autre état. La mémoire de notre *plotter* contient:


| Case de la mémoire | Au démarrage | Commandes l'affectant |
|---|---|---|
| **Position** (X, Y) | (0, 0) | `G0`, `G1`, `G2`, `G3` |
| **Hauteur Z** (stylo levé / posé) | levé | `G0` ou `G1` avec un `Z` |
| **Unités** (mm ou pouces) | millimètres | `G21` / `G20` |
| **Mode de coordonnées** (absolu / relatif) | absolu | `G90` / `G91` |
| **Vitesse** `F` | aucune | `F` |
| **Mode de déplacement** (G0, G1…) | aucun | `G0`, `G1`, `G2`, `G3` |


<div style="border-left:6px solid #2e8b57;background:rgba(46,139,87,.13);padding:4px 18px;border-radius:6px;margin:8px 0">

**💡 Ce que ça implique**

1. **Un trait part toujours de là où l'on est**, jamais de nulle part : la position de départ est simplement la position en mémoire.
2. **Les réglages sont persistants,** tout réglage reste valable **jusqu'à ce qu'on le change**. On peut donc ne pas le répéter à chaque ligne.
3. Un programme commence **toujours par une séquence d'initialisation:** on ne suppose pas l'état dans lequel la machine a été laissée par le programme précédent

</div>

Ainsi, le programme suivant est valide : plusieurs lignes n'indiquent pas de commande de déplacement, la machine réutilise la dernière. **Lisez-le** et prédisez la forme dessinée avant d'exécuter la cellule.

In [ ]:
programme3 = """G21 G90
G0 Z5
G0 X20 Y20
G1 Z0 F1000
G1 X80 Y20      ; ligne avec G1 : la machine retient « mode = G1 »
X80 Y60         ; pas de G ici : la machine réutilise G1
X20             ; pas de Y ici : la machine garde Y60
Y20             ; pas de X ici : la machine garde X20
G0 Z5
M2"""

show(programme3)

... mais rend le programme plus difficile à relire, car il faut sans cesse relire les instructions précédentes pour savoir quel mode est actif (trivial pour un *plotter*, plus aventureux pour une fraiseuse 5-axes)

<div style="border-left:6px solid #c25e00;background:rgba(194,94,0,.13);padding:4px 18px;border-radius:6px;margin:8px 0">

**⚠️ Ecrire des instructions complètes**

**Dans cet atelier, on écrit toujours la commande complète sur chaque ligne.** La clarté vaut mieux que quelques caractères économisés. Les paramètres machine, comme la vitesse, peuvent être omis.

</div>

## I.4. Une recette et des conventions

Tous les programmes de cet atelier suivent le même squelette, qui applique directement ce que nous venons de voir : on initialise la machine, on place le stylo *avant* de le poser, on trace, puis on le relève. Vous pouvez utiliser ce **modèle de départ** pour tous les exercices.

```
; Initialisation
G21 G90            1. initialisation : millimètres, positions absolues
G0 Z5              2. lever le stylo
; Dessin (répéter le bloc autant de fois par formes séparées)
G0 X.. Y..         3. aller au point de départ (stylo levé : on ne dessine pas)
G1 Z0 F..          4. poser le stylo (et régler la vitesse)
G1 X.. Y..         5. tracer : autant de lignes G1 que nécessaire
G0 Z5              6. lever le stylo
; Fin
M2                 7. fin du programme
```

<div style="border-left:6px solid #8e44ad;background:rgba(142,68,173,.11);padding:4px 18px;border-radius:6px;margin:8px 0">

**🧪 Exp.2. Retour en arrière**

Reprenez le premier programme de l'atelier et retrouvez les 7 étapes.

</div>

<div style="border-left:6px solid #2e8b57;background:rgba(46,139,87,.13);padding:4px 18px;border-radius:6px;margin:8px 0">

**💡 Les conventions de ce traceur**

- L'**origine** `(0, 0)` est le coin **en bas à gauche** de la feuille.
- **X** mesure la position **horizontale**, de la gauche vers la droite (→).
- **Y** mesure la position **verticale**, du bas vers le haut (↑).
- Toutes les distances sont en **millimètres**.
- On donne **toujours X avant Y** : « `X80 Y30` » veut dire « 80 mm vers la droite, 30 mm vers le haut ».
- **Z** est la **hauteur** de l'outil. Ici : `Z5` = stylo levé de 5 mm au-dessus du papier ; `Z0` = stylo posé sur le papier.

</div>

<div style="border-left:6px solid #4a6fa5;background:rgba(74,111,165,.10);padding:4px 18px;border-radius:6px;margin:8px 0">

**ℹ️ Que représente l'axe Z ?**

| Machine | Que représente Z ? | Valeurs typiques |
|---|---|---|
| **Traceur** (notre cas) | hauteur du stylo | `Z5` levé · `Z0` posé |
| **Fraiseuse** | profondeur de l'outil dans la matière | `Z5` au-dessus · `Z-3` = 3 mm *dans* la matière |
| **Imprimante 3D** | hauteur de la couche en cours | `Z0.2` première couche, puis `Z0.4`, `Z0.6`… |

Sur notre traceur, on ne descend jamais sous 0 : le papier est la « matière ». Sur une fraiseuse, `Z0` est la surface de la pièce et **Z négatif = on entre dans la matière** (c'est pour cela qu'elle travaille à `Z-3`). 

</div>

# II. (Exercices) Lire et Ecrire du Gcode

## II.1. Quatre programmes à lire

<div style="border-left:6px solid #d4a017;background:rgba(255,196,0,.17);padding:4px 18px;border-radius:6px;margin:8px 0">

**✏️ Énoncé**

Chacun des programmes suivants dessine une forme. Laquelle ?

Pour chaque programme : **(1)** lisez-le sans l'exécuter, en suivant le stylo sur un brouillon ; **(2)** notez votre réponse ; **(3)** exécutez la cellule pour vérifier ;
**(4)** consultez les réponses en bas de section.

</div>

**Question 1:** Un carré, un cercle, un losange ?

In [ ]:
# 1. Un carré, un cercle, un losange ?
prog1 = """
G21 G90
G0 Z5
G0 X30 Y40
G1 Z0 F1000
G1 X80 Y40
G1 X80 Y90
G1 X30 Y90
G1 X30 Y40
G0 Z5
M2
"""

# show(prog1)

**Question 2:** Un carré, un cercle, un losange ?

In [ ]:
# 2. Un carré, un cercle, un losange ?
prog2 = """
G21 G90
G0 Z5
G0 X60 Y30
G1 Z0 F1000
G1 X100 Y60
G1 X60 Y90
G1 X20 Y60
G1 X60 Y30
G0 Z5
M2
"""

# show(prog2)

**Question 3:** un cercle, un coeur, une porte dessinée en plan ?

In [ ]:
# 3. Un cercle, un cœur, une porte dessinée en plan (quart de cercle) ?
prog3 = """
G21 G90
G0 Z5
G0 X30 Y30
G1 Z0 F1000
G1 X30 Y90
G2 X90 Y30 I0 J-60
G1 X30 Y30
G0 Z5
M2
"""

# show(prog3)

**Question 4:** Un trèfle, un coeur, un carreau, un pique ?

In [ ]:
# 4. Un trèfle, un coeur, un carreau, un pique ?
prog4 = """
G21             ; unites en mm
G90             ; coordonnees absolues

G0 Z5           ; releve
G0 X50 Y10      ; pointe
G1 Z0 F300     ; descente

G1 X21.716 Y38.284 F1000                ; cote gauche, montee a 45 degres
G2 X50 Y66.569 I14.142 J14.142          ; demi-cercle gauche (sens horaire)
G2 X78.284 Y38.284 I14.142 J-14.142     ; demi-cercle droit (sens horaire)
G1 X50 Y10                              ; cote droit, retour a la pointe

G0 Z5           ; releve
G0 X0 Y0        ; retour origine
M2              ; fin de programme
"""

# show(prog4)

**Question 5 :** On supprime la ligne `G0 X20 Y20` (la ligne n°3) du premier programme, celui de la section I.1. Que va-t-il se passer ?

- **(A)** rien ne change : cette ligne ne sert à rien
- **(B)** la machine s'arrête avec une erreur, car il manque une instruction
- **(C)** le trait part d'un endroit inattendu : l'origine de la machine
- **(D)** tout le dessin est simplement décalé, sans changer de forme

**Réponses**

<details>
<summary>✅ <i>Question 1</i></summary>

Le stylo va de (30, 40) à (80, 40), puis (80, 90), (30, 90) et revient à (30, 40) : quatre côtés, tous parallèles aux axes. La largeur (80 − 30) et la hauteur (90 − 40) valent
toutes deux 50 mm : c'est donc un **carré**, pas un rectangle quelconque. Il n'y a ni `G2` ni `G3`, donc aucun arc : ce n'est pas un cercle.

</details>

<details>
<summary>✅ <i>Question 2</i></summary>

Aucun côté n'est parallèle aux axes : le stylo passe par (60, 30), (100, 60), (60, 90), (20, 60) puis revient au départ. Chaque côté avance de 40 mm en X et de 30 mm en Y,
soit une longueur de 50 mm (c'est un triangle 3-4-5) : quatre côtés égaux. Les diagonales mesurent 80 mm et 60 mm : elles sont inégales, donc les angles ne sont pas droits.
Ce n'est pas un carré, mais un **losange**.

</details>

<details>
<summary>✅ <i>Question 3</i></summary>

Le trait de (30, 30) à (30, 90) est le **battant** de la porte ; l'arc `G2` en est le **débattement**, le quart de cercle que décrit le battant quand on l'ouvre. Son centre est
la charnière : départ (30, 90) + `I0 J-60` = (30, 30), et le rayon vaut 60 mm. La dernière ligne `G1 X30 Y30` ferme la figure en rejoignant la charnière.
Ce n'est pas un cercle : l'arc ne couvre qu'un quart de tour (de 90° à 0°).

</details>

<details>
<summary>✅ <i>Question 4</i></summary>

Le stylo part de la pointe (50, 10), monte à 45° jusqu'à (21.7, 38.3), puis décrit deux demi-cercles `G2` (sens horaire) de rayon 20 mm : celui de gauche passe par le haut
à gauche, celui de droite par le haut à droite. Il redescend enfin à 45° vers la pointe. Remarquez les deux premières lignes de réglage supplémentaires, `G17` et `G94`,
fréquentes dans les programmes issus d'autres logiciels : elles ne changent rien ici. Le stylo est posé à `Z-1` (donc un peu « appuyé ») : le simulateur le considère posé.

</details>

<details>
<summary>✅ <i>Question 5</i></summary>

Sans la ligne `G0 X20 Y20`, la machine n'a pas bougé depuis le démarrage : elle est toujours à l'**origine (0, 0)**. Quand `G1 Z0` pose le stylo, il se pose donc à l'origine,
et le trait `G1 X80 Y50` part de là au lieu de partir de (20, 20). C'est l'application directe de la règle vue en I.3 : *un trait part toujours de là où l'on est*.
Ce n'est pas (A), car le dessin change ; pas (B), car la machine ne détecte aucune erreur (le programme est valide, simplement différent) ; pas (D), car le dessin n'est pas décalé : son point d'arrivée reste en (80, 50), seul son point de départ change. **La (C) !**

</details>

## II.2. Deux quadilatères à dessiner

### II.2.1. Un carré de 40mm à trous

<div style="border-left:6px solid #d4a017;background:rgba(255,196,0,.17);padding:4px 18px;border-radius:6px;margin:8px 0">

**✏️ Enoncé**

Complétez ce programme à trous pour dessiner un **carré de 40 mm de côté**, dont le coin inférieur gauche est en `(10, 10)`.

Les `___` sont à remplacer par des nombres ou des commandes. Le premier côté (celui du bas) est déjà écrit : il reste trois côtés à tracer, en tournant dans le sens anti-horaire.

</div>

In [ ]:
gcode = """
G21 G90
G0 Z5
G0 X10 Y10             ; aller au coin bas-gauche
G1 Z0 F500             ; poser le stylo
G1 X50 Y10 F2000       ; 
G1 X___ Y___           ; 
G1 X___ Y___           ; 
G1 X___ Y___           ; 
G0 Z___                ; lever le stylo
M2"""

show(gcode)

<div style="border-left:6px solid #4a6fa5;background:rgba(74,111,165,.10);padding:4px 18px;border-radius:6px;margin:8px 0">

**ℹ️ Pourquoi deux vitesses `F` ?**

`F` est la vitesse, en mm par minute. On descend doucement le stylo (`F500`), puis on trace plus vite (`F2000`). Une fois réglée à `F2000`, ce paramètre reste valable pour toutes les lignes suivantes.

</div>

<details>
<summary>🔎 <i>Indice 1</i></summary>

Un carré a quatre côtés. La ligne `G1 X50 Y10` en trace déjà un : le côté du bas, de gauche à droite. Pour chacune des trois lignes suivantes, posez-vous la question :
*« dans quel coin dois-je arriver ? »* On tourne dans le sens anti-horaire : côté de droite (vers le haut), côté du haut (vers la gauche), côté de gauche (vers le bas).

</details>

<details>
<summary>🔎 <i>Indice 2</i></summary>

Les quatre coins du carré sont `(10, 10)`, `(50, 10)`, `(50, 50)` et `(10, 50)`. Après `(50, 10)`, on va donc en `(50, 50)`, puis en `(10, 50)`, et la dernière ligne
doit revenir au point de départ `(10, 10)` pour fermer la figure. Pour la dernière ligne à trou, souvenez-vous de la hauteur du stylo levé (Z5).

</details>

<details>
<summary>✅ <i>Solution</i></summary>

```
G21 G90
G0 Z5
G0 X10 Y10             ; aller au coin bas-gauche (stylo levé)
G1 Z0 F500             ; poser le stylo, doucement
G1 X50 Y10 F2000       ; côté du bas, à la vitesse de travail
G1 X50 Y50             ; côté de droite : monter jusqu'au coin haut-droit
G1 X10 Y50             ; côté du haut : aller jusqu'au coin haut-gauche
G1 X10 Y10             ; côté de gauche : revenir au point de départ
G0 Z5                  ; lever le stylo
M2
```

</details>

###  II.2.2. Un rectangle

<div style="border-left:6px solid #d4a017;background:rgba(255,196,0,.17);padding:4px 18px;border-radius:6px;margin:8px 0">

**✏️ Enoncé**

Dessiner un **rectangle de 80 mm de large et 30 mm de haut**, dont le coin inférieur gauche est en `(20, 20)`, en s'aidant du modèle et du programme précédent

</div>

In [ ]:
gcode = """
; à vous d'écrire le programme, en suivant le modèle de la section I.4
"""

show(gcode)
check(gcode, "ex2b")

<details>
<summary>🔎 <i>Indice 1</i></summary>

Reprenez le programme du carré : la structure est identique, seules les coordonnées changent. Pour trouver les quatre coins, partez du coin bas-gauche `(20, 20)`,
puis ajoutez la **largeur** (80) à X ou la **hauteur** (30) à Y.

</details>

<details>
<summary>🔎 <i>Indice 2</i></summary>

Les quatre coins sont `(20, 20)`, `(100, 20)`, `(100, 50)` et `(20, 50)`. Le trait fait le tour dans cet ordre, puis revient en `(20, 20)`.

</details>

<details>
<summary>✅ <i>Solution</i></summary>

```
G21 G90
G0 Z5
G0 X20 Y20             ; coin bas-gauche
G1 Z0 F500
G1 X100 Y20 F2000      ; côté du bas : 80 mm
G1 X100 Y50            ; côté de droite : 30 mm
G1 X20 Y50             ; côté du haut : 80 mm
G1 X20 Y20             ; côté de gauche : 30 mm
G0 Z5
M2
```

</details>

## II.3. Déboguage

Déboguer consiste à retrouver et corriger les erreurs d'un programme. C'est une activité à part entière, que l'on pratique tout le temps. La bonne méthode : **observer** (la figure, les messages), **émettre une hypothèse**, **corriger une seule chose**, **ré-exécuter**.

<div style="border-left:6px solid #d4a017;background:rgba(255,196,0,.17);padding:4px 18px;border-radius:6px;margin:8px 0">

**✏️ Enoncé**

Ce programme devrait dessiner un carré et un triangle. Retrouver et corriger les **trois erreurs**. 

</div>

In [ ]:
gcode = """G21 G90
G0 Z5
G0 X20 Y20
G1 Z0 F800
G1 X60 Y20
G1 X60 Y60
G1 X20 Y6
G1 X20 Y20
G0 X80 Y20
G1 Z0
G1 X110 20
G1 X95 Y45
G1 X80 Y20
G0 Z5
M2"""

show(gcode)
# show(gcode, target="ex4")
check(gcode, "ex4")

<details>
<summary>🔎 <i>Indice 1</i></summary>

Les trois erreurs sont de natures différentes : une faute de **syntaxe** (le simulateur la signale en rouge, avec son numéro de ligne), une faute de **valeur**
(une forme déformée, qui ne colle pas au trait vert) et une **instruction oubliée** (un trait apparaît là où il ne devrait pas y en avoir).

</details>

<details>
<summary>🔎 <i>Indice 2</i></summary>

Procédez dans cet ordre, en ré-exécutant après chaque correction. D'abord le message rouge : que manque-t-il dans la ligne désignée ? Ensuite, comparez la forme du carré
avec le trait vert : quel côté est de travers, et quel nombre en est responsable ? Enfin, regardez le message orange : que fait le stylo quand la machine passe du carré au triangle ?

</details>

<details>
<summary>✅ <i>Solution</i></summary>

```
G21 G90
G0 Z5
G0 X20 Y20
G1 Z0 F800
G1 X60 Y20
G1 X60 Y60
G1 X20 Y60       ; ✔ erreur n°2 : il y avait écrit Y6 au lieu de Y60
G1 X20 Y20
G0 Z5            ; ✔ erreur n°3 : ligne manquante, il fallait lever le stylo avant de partir vers le triangle
G0 X80 Y20
G1 Z0
G1 X110 Y20      ; ✔ erreur n°1 : la lettre Y était manquante
G1 X95 Y45
G1 X80 Y20
G0 Z5
M2
```
Les erreurs sont numérotées dans l'ordre où on les rencontre en déboguant : **n°1** la syntaxe (message rouge du simulateur), **n°2** la valeur fausse (le carré est déformé), **n°3** l'oubli
(sans `G0 Z5`, le `G0 X80 Y20` se fait stylo posé : il traîne un trait parasite entre le carré et le triangle, ce que signale l'avertissement orange).

</details>

## II.4. Absolu et Relatif, `G90` et `G91`

Revenons sur les paramètres de la machine. Deux modes de déplacement sont paramétrables, et la machine en retient un à la fois:
- **Absolu** (`G90`) : les coordonnées sont une **position sur la feuille**, mesurée depuis l'origine. C'est le mode dans lequel on a travaillé jusqu'ici.
- **Relatif** (`G91`) : chaque paire de coordonnées est un **déplacement** à partir de la position actuelle.


| | Absolu (`G90`) | Relatif (`G91`) |
|---|---|---|
| `G1 X25 Y0` signifie… | « va au point X25 Y0 » | « avance de 25 mm vers la droite » |
| Avantage | chaque ligne est **indépendante** des autres ; une erreur reste locale | pratique pour **répéter un motif** (escalier, trame, grille) |
| Inconvénient | pour répéter un motif, il faut recalculer toutes les positions | une erreur se **propage** à toutes les lignes suivantes ; l'ordre est critique |

Pour en apprécier la différence, voici deux programmes ci-dessous contenant **exactement les mêmes lignes de dessin** (`G1 X25 Y0` puis `G1 X0 Y15`). Seul le mode change.

In [ ]:
absolu = """G21 G90
G0 Z5
G0 X10 Y10
G1 Z0 F1500
G1 X25 Y0         ; en absolu : « va au point X25 Y0 »
G1 X0 Y15         ; en absolu : « va au point X0 Y15 »
G0 Z5
M2"""

relatif = """G21 G90
G0 Z5
G0 X10 Y10
G1 Z0 F1500
G91               ; ← on passe en relatif
G1 X25 Y0         ; en relatif : « avance de 25 mm vers la droite »
G1 X0 Y15         ; en relatif : « monte de 15 mm »
G90               ; ← retour en absolu
G0 Z5
M2"""

show(absolu)
show(relatif)

<div style="border-left:6px solid #d4a017;background:rgba(255,196,0,.17);padding:4px 18px;border-radius:6px;margin:8px 0">

**✏️ Énoncé**

Dessiner un escalier de **4 marches**, chaque marche faisant **25 mm** de long et **15 mm** de haut, en partant du point `(10, 10)`, en **déplacements relatifs**.

Le programme à trous passe en relatif, puis repasse en absolu. Remplacez la ligne `...` par les lignes de l'escalier et le `___` par la commande qui convient.

</div>

In [ ]:
gcode = """
G21 G90
G0 Z5
G0 X10 Y10
G1 Z0 F1500
G91                       ; ← passage en relatif
...                       ; plusieurs lignes pour dessiner l'escalier
___                       ; ← retour en absolu
G0 Z5
M2"""

show(gcode)
# show(gcode, target="ex5a")
check(gcode, "ex5a")

<details>
<summary>🔎 <i>Indice 1</i></summary>

Une marche, c'est un déplacement de 25 mm vers la droite (`G1 X25 Y0`), puis une contremarche de 15 mm vers le haut (`G1 X0 Y15`). En relatif, `X0` ou `Y0` veut simplement dire
« ne bouge pas selon cet axe ». Il y a 4 marches, soit 8 lignes à écrire à la place de `...`.

</details>

<details>
<summary>🔎 <i>Indice 2</i></summary>

Les deux lignes de la première marche se répètent **à l'identique** pour les trois suivantes : c'est tout l'intérêt du mode relatif. Quant au `___`, c'est la commande qui
remet la machine en positions absolues ; sans elle, le `G0 Z5` suivant serait lu comme un déplacement relatif.

</details>

<details>
<summary>✅ <i>Solution</i></summary>

```
G21 G90
G0 Z5
G0 X10 Y10
G1 Z0 F1500
G91                       ; ← à partir d'ici, les nombres sont des DÉPLACEMENTS
G1 X25 Y0                 ; marche 1
G1 X0 Y15                 ; contremarche 1
G1 X25 Y0                 ; marche 2
G1 X0 Y15                 ; contremarche 2
G1 X25 Y0                 ; marche 3
G1 X0 Y15                 ; contremarche 3
G1 X25 Y0                 ; marche 4
G1 X0 Y15                 ; contremarche 4
G90                       ; ← retour en positions absolues
G0 Z5
M2
```

</details>

<div style="border-left:6px solid #8e44ad;background:rgba(142,68,173,.11);padding:4px 18px;border-radius:6px;margin:8px 0">

**🧪 Exp.3. Un autre escalier ?**

En fin de programme, la machine repasse en mode position absolu. Modifiez votre programme pour dessiner un **second escalier** : déplacez d'abord le stylo (stylo levé, en absolu) à un autre endroit
de la feuille, repassez en relatif, puis recopiez les huit lignes de l'escalier par un grand copier-coller.

</div>

<div style="border-left:6px solid #2e8b57;background:rgba(46,139,87,.13);padding:4px 18px;border-radius:6px;margin:8px 0">

**💡 Règle d'or**

**Par défaut, ravaillez en absolu.** Passez en relatif **seulement** pour un motif répété, et **revenez en absolu dès que vous avez fini** (`G90`). Un `G91` oublié fausse silencieusement **toutes** les lignes suivantes : la machine n'a aucun moyen de savoir que vous aviez changé d'avis.

</div>

## II.5. Suivez le plan

<div style="border-left:6px solid #d4a017;background:rgba(255,196,0,.17);padding:4px 18px;border-radius:6px;margin:8px 0">

**✏️ Énoncé**

Voici le plan d'une **petite pièce rectangulaire**, avec **une seule porte** dans le mur du bas. À partir de ce dessin, rédigez les instructions machine pour le reproduire :
lire les coordonnées sur la grille, décider de l'ordre des tracés, écrire les lignes. **Le code de la porte vous est donné en indice** ; à vous d'écrire celui des murs.

</div>

<div style="border-left:6px solid #2e8b57;background:rgba(46,139,87,.13);padding:4px 18px;border-radius:6px;margin:8px 0">

**💡 Stratégie : écrire un peu, tester, et écrire un peu**

1. **Relevez les coordonnées** des coins sur la grille et notez-les sur un brouillon.
2. **Décidez de l'ordre.** Combien de fois faut-il lever le stylo ? Les murs forment-ils un trait continu ? Où commence-t-il, où finit-il ?
3. **Écrivez le début**, exécutez, regardez. Puis ajoutez quelques lignes, exécutez, regardez. Ne cherchez pas à tout écrire d'un coup : une erreur est bien plus facile à trouver dans 5 lignes que dans 25.

</div>

In [ ]:
show_target("ex7", labels=False, grid=10)

In [ ]:
gcode = """

; Les murs

; La porte

M2
"""

show(gcode, target="ex7", grid=10)
check(gcode, "ex7")

<details>
<summary>🔎 <i>Indice Porte</i></summary>

```
G0 Z5                 ; (au cas où le stylo serait resté posé)
G0 X50 Y20            ; aller à la charnière de la porte
G1 Z0 F1000           ; poser le stylo
G1 X50 Y50            ; le battant : de la charnière jusqu'à son extrémité (ouvert à 90°)
G2 X80 Y20 I0 J-30    ; l'arc de débattement, centré sur la charnière, en sens horaire
G0 Z5                 ; lever le stylo
```
</details>

<details>
<summary>🔎 <i>Indice 1</i></summary>

Les murs forment **un seul trait continu**, qui s'arrête de part et d'autre de la porte : le mur du bas est interrompu par l'ouverture. Choisissez l'une des extrémités de l'ouverture comme
point de départ, faites le tour de la pièce, et terminez à l'autre extrémité. Les murs forment un rectangle : combien de coins faut-il atteindre ?

</details>

<details>
<summary>🔎 <i>Indice 2</i></summary>

Sur la grille, la pièce occupe de `X20` à `X120` et de `Y20` à `Y90`. L'ouverture de la porte va de `X50` à `X80` sur le mur du bas : ce sont exactement la charnière `(50, 20)` et la pointe de l'arc `(80, 20)`
du code fourni. On peut donc partir de `(50, 20)`, aller vers la gauche jusqu'au coin `(20, 20)`, puis monter, traverser, redescendre, et s'arrêter en `(80, 20)`.

</details>

# III. Automatiser

Les instructions *GCode* sont du texte brut: un langage machine simple, dont chaque ligne est destinée au *firmware* de la machine-outil. Toutefois, écrire les instructions à la main revient à penser en **mouvements et paramètre de machine** (lever le stylo, déplacer la broche, changer de mode d'extrusion...) plutôt qu'en **géométrie**. D'autant plus que l'écriture d'instruction devient fastidieux dès que la machine-outil possède plus de 4 axes, que la géométrie souhaitée est complexe, que les phases d'usinages s'enchainent (eg. ébauche/opérations/finitions en fraisage) ou tout simplement, que la géométrie souhaitée est répétitive.

Pour s'extraire d'un langage machine [bas-niveau](https://fr.wikipedia.org/wiki/Langage_de_programmation_de_bas_niveau), dont la programmation requiert une attention sur la gestion de la machine et ses contraintes, une solution est d'utiliser un langage de programmation de **haut-niveau**. Par un ensemble d'abstractions, ce dernier permet de se concentrer sur le problème - ici, dessiner une géométrie plane à l'aide d'une MOCN - plutôt que de se concentrer sur la machine en elle-même. En règle générale: ils plus rapides et plus faciles à écrire autant qu'à déboguer ; ils sont plus facilement portables d'une machine à une autre.

```text
géométrie  ──►  postprocesseur  ──►  GCode  ──►  firmware  ──►  machine
(tracés)    (programme traducteur)   (texte)
```

Selon son rôle, ce programme:
+ **permet de tracer la géométrie**, cela va d'un simple *script* à un *logiciel* avec une *interface* complète comme Rhino ou Autocad
+ **postprocesser une géométrie**, c'est à dire traduire une géométrie en GCode pour une machine. Le rôle du postprocesseur est de "penser machine". C'est ce que sont les [slicer](https://fr.wikipedia.org/wiki/Logiciel_de_d%C3%A9coupage_en_tranches) pour l'impression 3d et autres modules CAM.

En partie III, nous commencerons par écrire du *GCode* répétitif à l'aide d'un programme ; puis nous utiliserons un postprocesseur qui traduira en *GCode* une géométrie `.svg`
1. **Hachures** : le GCode reste du texte, mais on l'écrit avec un programme ; variables, listes et boucles servent à produire les instructions à répéter ;
2. **Mosaïque / motif** : on dessine de la géométrie au format `.svg` avec un module Python, et un postprocesseur la traduit en GCode.

## III.1. Hachures, écrire des lignes de GCode

<div style="border-left:6px solid #14365d;background:rgba(20,54,93,.10);padding:4px 18px;border-radius:6px;margin:8px 0">

**🎯 Objectif**

Dessiner une série de 11 hachures verticales espacées de 4mm, à l'aide de quelques lignes de *python* qui écrivent le *GCode* à notre place.

</div>

### Trois notions de programmation en 2 minutes

**1. Variables.** En programmation, une **variable** associe un nom à une valeur. Ci-dessous, nous assignons des valeurs à trois variables : `x`, `y` et `ligne`.

In [ ]:
x = 10                  # un nombre entier
y = 30                  # un nombre entier
ligne = f"G0 X{x} Y{y}" # une `f-string`: un modèle de chaine de caractère où chaque `{}` est remplacé par la variable correspondante
print(ligne)            # print() est une fonction qui affiche la variable correspondante. Ici `ligne`

<div style="border-left:6px solid #8e44ad;background:rgba(142,68,173,.11);padding:4px 18px;border-radius:6px;margin:8px 0">

**🧪 Exp.4. Changement de variables**

Par ce petit programme Python, nous avons fabriqué une ligne de *GCode* déplaçant le stylo vers une position donnée par deux variables `x` et `y`. Changez `x` ou `y`, ré-exécutez : la ligne de GCode change avec eux.

</div>

**2. Boucles.** Une boucle permet de répéter un ensemble d'instructions, selon une condition préétablie. Dans cette boucle `for`, la variable `k` prend successivement les valeurs 0, 1, 2, 3.

In [ ]:
for k in range(4):  # range() est une fonction qui produit une liste d'entiers 
    print(f"G0 X{10 * k} Y0")

**3. Listes.** Une liste est une suite ordonnée d'éléments, écrite entre crochets. On ajoute un élément à la fin avec `.append(...)`, et `"\n".join(liste)` fusionne tous les éléments en un seul texte, un élément par ligne, suivant le format du *GCode*.

In [ ]:
lignes = ["G21 G90", "G0 Z5"]     # une liste de deux lignes de GCode
lignes.append("G0 X10 Y10")       # ajoute un élément à la fin de la liste
lignes.append("M2")

print(lignes)                     # la liste, avec ses crochets et ses guillemets
print("\n".join(lignes))          # le programme : une ligne de GCode par élément

<div style="border-left:6px solid #8e44ad;background:rgba(142,68,173,.11);padding:4px 18px;border-radius:6px;margin:8px 0">

**🧪 Exp.5. Changement de variables II**

Le programme ci-dessous écrit des hachures : une **boucle** (`for`) répète 11 fois les instructions d'une hachure et les ajoute à une **liste** de lignes, assemblée en un texte GCode à la fin. Changez les valeurs de `nombre_de_hachures` et `espacement`, puis appréciez de ne rien avoir à écrire d'autre pour produire plus ou moins de hachures.

Ce dessin de hachures pourrait-il être paramétré par d'autres variables ? Lesquelles ?

</div>

In [ ]:
nombre_de_hachures = 11    # combien de hachures ?
espacement = 4             # distance (en mm) entre deux traits consécutifs

# Initialisation de la liste `lignes` par les deux premières instructions de notre modèle
lignes = [
    "G21 G90", 
    "G0 Z5"
]

for k in range(nombre_de_hachures):
    y = 10 + espacement * k             # calcul de la valeur de `y` d'une hachure
    lignes.append(f"G0 X10 Y{y}")       # déplacement au début de la ligne a X10
    lignes.append("G1 Z0 F1500")        # pose du stylo
    lignes.append(f"G1 X50 Y{y}")       # tracé de la hachure jusque X50
    lignes.append("G0 Z5")              # levé du stylo
lignes.append("M2")

gcode = "\n".join(lignes)               # conversion de la liste d'instructions en une chaine de caractères
show(gcode, grid=10)

## III.2. Mosaïque, de la géométrie au GCode (svg.py)

Les hachures sont écrites par un programme, mais restent proches de la machine : on fabrique des lignes de GCode, avec leurs `G0`, `G1` et `Z`. Nous allons maintenant nous abstraire davantage : décrire des **formes** (rectangle, cercle, polygone) dans un format de dessin vectoriel, le **SVG**, puis laisser un **postprocesseur** écrire le GCode correspondant.

Le **SVG** (*Scalable Vector Graphics*) est un format de dessin vectoriel lu par les navigateurs et par la plupart des logiciels de dessin et de CAO (Illustrator, Inkscape, Rhino…). Il est structuré par des **balises** dans un texte, où chaque forme est décrite par une balise distincte (eg. `<rect ...>`, `<circle ...>`). 

La bibliothèque Python `svg` construit ces balises, forme par forme, à notre place, à partir d'une liste de géométries propres à cette bibliothèque.

### III.2.1. Dessiner avec un module SVG

Setup

In [ ]:
# install modules
# !pip install svg.py
# !uv add svg.py

In [ ]:
# import, utils
import svg 

def assemble_svg_elements_to_canvas(elements: list[svg.Element]) -> svg.SVG:
    AFFICHAGE = "width:100%; max-width:720px; height:auto; border:1px solid #bbb"
    canvas = svg.SVG(
        width="297mm", height="210mm",              # la feuille A4 paysage, en millimètres
        viewBox=svg.ViewBoxSpec(0, 0, 297, 210),    # 1 unité de dessin = 1 mm
        style=AFFICHAGE,
        elements=elements
    )
    return canvas

Quatres éléments

In [ ]:
quatre_traces_elements = [
    svg.Rect(x=20, y=20, width=40, height=30, fill="none", stroke="black"),
    svg.Circle(cx=100, cy=35, r=15, fill="none", stroke="black"),
    svg.Polygon(points=[140, 50, 160, 20, 180, 50], fill="none", stroke="black"),
    svg.Line(x1=200, y1=20, x2=240, y2=50, stroke="black"),
]   # liste de quatre tracés

quatre_traces_svg = assemble_svg_elements_to_canvas(elements=quatre_traces_elements)

print(str(quatre_traces_svg).replace("><", ">\n<"))       # le SVG est du texte : une balise par forme
quatre_traces_svg                                         # et voici son rendu

<div style="border-left:6px solid #2e8b57;background:rgba(46,139,87,.13);padding:4px 18px;border-radius:6px;margin:8px 0">

**💡 Conventions du SVG**

- L'**origine** `(0, 0)` est le coin **en haut à gauche** de la feuille, et **Y croît vers le bas**. C'est l'inverse de notre traceur (origine en bas à gauche, Y vers le haut).
- Les unités sont fixées par `viewBox` : ici 1 unité = 1 mm, comme pour la machine.
- `stroke` est la couleur du **contour**, `fill` celle du **remplissage**. Un traceur ne sait que tracer des contours : seuls les contours nous intéressent (`fill="none"`).

</div>

<div style="border-left:6px solid #8e44ad;background:rgba(142,68,173,.11);padding:4px 18px;border-radius:6px;margin:8px 0">

**🧪 Exp.5. Des formes, et une référence**

Reprenez le programme ci-dessus. Ajoutez ou supprimez des formes de la liste `quatre_traces_elements`. 

Malheureusement, cette bibliothèque n'est pas bien documentée, ni les paramètres des objets (les formes `svg.Rect`, `svg.Circle`), ni les fonctions ne sont détaillées. Le créateur de la bibliothèque nous envoie regarder les [exemples](https://github.com/orsinium-labs/svg.py/tree/master) plutôt que de nous transmettre une *référence* en bonne et due forme.

</div>

### III.2.2. Le postprocesseur

Un **postprocesseur** est un programme qui lit une géométrie et écrit les instructions de la machine. Celui ci est minimal, il parcourt les formes SVG et, pour chacune, écrit son **contour**, gère le stylo et les vitesses de trajet.

| Forme SVG | Contour en GCode |
|---|---|
| `rect` | quatre côtés `G1`, avec retour au point de départ |
| `line` | un seul `G1` |
| `polygon` / `polyline` | un `G1` par sommet (et, pour `polygon`, retour au point de départ) |
| `circle` | un seul **arc** `G2`, dont l'arrivée est le départ (cercle complet) et `I = -r` : le centre est à `r` à gauche du départ |

Exécutez-la cellule ci dessous, elle **définit** la fonction `svg2plottergcode()` que nous utiliserons pour postprocesser la géométrie svg.

In [ ]:
import re
import xml.etree.ElementTree as ET

def svg2plottergcode(svg_text: str, feed: int =1500) -> str:
    """Postprocesseur minimal : contours des formes de base d'un SVG (1 unité = 1 mm) -> GCode."""
    racine = ET.fromstring(svg_text)                         # lecture du texte SVG
    H = float(racine.get("viewBox").split()[3])              # hauteur de la feuille (le SVG compte Y vers le bas)
    P = lambda x, y: f"X{round(x, 3)} Y{round(H - y, 3)}"    # un point SVG -> un point machine (Y retourné)
    g = ["G21 G90", "G0 Z5"]                                 # initialisation, comme toujours
    for e in racine.iter():                                  # on parcourt tous les éléments du SVG
        forme, attr = e.tag.split("}")[-1], e.attrib         # le nom de la forme, et ses attributs
        num = lambda k: float(attr[k])
        if forme == "rect":
            x, y, w, h = num("x"), num("y"), num("width"), num("height")
            pts = [(x, y), (x + w, y), (x + w, y + h), (x, y + h), (x, y)]
        elif forme == "line":
            pts = [(num("x1"), num("y1")), (num("x2"), num("y2"))]
        elif forme in ("polygon", "polyline"):
            v = [float(t) for t in re.findall(r"-?[\d.]+", attr["points"])]
            pts = list(zip(v[::2], v[1::2])) + ([] if forme == "polyline" else [(v[0], v[1])])
        elif forme == "circle":                              # un cercle = un seul arc G2 (départ = arrivée)
            cx, cy, r = num("cx"), num("cy"), num("r")
            g += [f"G0 {P(cx + r, cy)}", f"G1 Z0 F{feed}", f"G2 {P(cx + r, cy)} I{-r} J0", "G0 Z5"]
            continue
        else:
            if forme not in ("svg", "g", "defs"):
                print(f"ignoré : <{forme}>")                 # forme non gérée : on le dit
            continue
        g += [f"G0 {P(*pts[0])}", f"G1 Z0 F{feed}"] + [f"G1 {P(*p)}" for p in pts[1:]] + ["G0 Z5"]
    return "\n".join(g + ["M2"])

Utilisons ce postprocesseur et voyons ce que ferait la machine

In [ ]:
quatre_traces_elements = [
    svg.Rect(x=20, y=20, width=40, height=30, fill="none", stroke="black"),
    svg.Circle(cx=100, cy=35, r=15, fill="none", stroke="black"),
    svg.Polygon(points=[140, 50, 160, 20, 180, 50], fill="none", stroke="black"),
    svg.Line(x1=200, y1=20, x2=240, y2=50, stroke="black"),
]   # liste de quatre tracés

quatre_traces_svg = assemble_svg_elements_to_canvas(elements=quatre_traces_elements)
gcode = svg2plottergcode(str(quatre_traces_svg))

print(gcode)

In [ ]:
show(gcode, grid=30)

<div style="border-left:6px solid #4a6fa5;background:rgba(74,111,165,.10);padding:4px 18px;border-radius:6px;margin:8px 0">

**ℹ️ Limites**

Ce postprocesseur ne connaît que les **formes de base** (`rect`, `line`, `polyline`, `polygon`, `circle`). Les autres éléments (`path`, `ellipse`, `text`...) sont signalés par un message « ignoré », et l'attribut `transform`
(rotation, translation…) n'est pas pris en compte.  
Comme le SVG exporté d'un logiciel de dessin est le plus souvent fait de `path` (courbes de Bézier), ce postprocesseur n'est pas adapté.

Un module qui *analyse* le SVG, comme `svgelements` (`!pip install svgelements`), traite tous ces cas en une quinzaine de lignes. Contrepartie : les courbes sont découpées en petits segments droits, donc le cercle n'est plus un arc `G2`.

</div>

<details>
<summary>🔎 <i>Version complète avec svgelements</i></summary>

```
import io
from svgelements import SVG, Shape, Path


def svg2gcode_complet(svg_text, feed=1500, pas=2.0):
    doc = SVG.parse(io.StringIO(svg_text), ppi=25.4)             # ppi=25.4 : 1 unité = 1 mm
    H = doc.viewbox.height
    P = lambda p: f"X{round(p.x, 3)} Y{round(H - p.y, 3)}"
    g = ["G21 G90", "G0 Z5"]
    for forme in doc.elements():
        if isinstance(forme, Shape):
            for sous_chemin in abs(Path(forme)).as_subpaths():   # abs() applique les transformations
                c = Path(sous_chemin)
                n = max(2, int(c.length() / pas))                # segments d'environ `pas` mm
                pts = [c.point(i / n) for i in range(n + 1)]
                g += [f"G0 {P(pts[0])}", f"G1 Z0 F{feed}"] + [f"G1 {P(p)}" for p in pts[1:]] + ["G0 Z5"]
    return "\n".join(g + ["M2"])
```

</details>

<div style="border-left:6px solid #8e44ad;background:rgba(142,68,173,.11);padding:4px 18px;border-radius:6px;margin:8px 0">

**🧪 Exp.6. (pour aller plus loin) Variations sur un motif**

Modifiez le programme de la mosaïque ci-dessous :
- ajoutez dans chaque tuile le **losange inscrit** (un `svg.Polygon` dont les quatre sommets sont les milieux des côtés) ;
- changez le nombre de colonnes et de lignes, le côté des tuiles.
- alternez en **damier** les cercles et les polygones. Deux indices: (1) nécessite une condition `if` / `else`, (2) une bonne manière de faire, `(i + j) % 2 == 0`

Notez ce qui n'a **jamais** eu besoin d'être touché : le postprocesseur. Une fois la traduction géométrie → GCode écrite, on ne pense plus qu'à la forme.

</div>

In [ ]:
cote = 30
x0, y0 = 20, 20

formes = []
for j in range(3):
    for i in range(4):
        x = x0 + cote * i
        y = y0 + cote * j
        formes.append(svg.Rect(x=x, y=y, width=cote, height=cote, fill="none", stroke="black"))
        formes.append(svg.Circle(cx=x + cote / 2, cy=y + cote / 2, r=cote / 2, fill="none", stroke="black"))   # le cercle inscrit

mosaique = svg.SVG(width="297mm", height="210mm", viewBox=svg.ViewBoxSpec(0, 0, 297, 210), elements=formes)
display(mosaique)

gcode = svg2plottergcode(str(mosaique))

In [ ]:
show(gcode, grid=30)

<div style="border-left:6px solid #2e8b57;background:rgba(46,139,87,.13);padding:4px 18px;border-radius:6px;margin:8px 0">

**💡 Pour aller plus loin**

- **Remplir** une forme : combiner ce chapitre et le précédent, en traçant des hachures à l'intérieur d'un contour.
- **Reprendre un vrai dessin** : exporter un plan simple en SVG depuis un logiciel de CAO, puis le passer à `svg2gcode_complet`.
- **Optimiser** le trajet : les déplacements à vide sont les pointillés de la figure. Réordonner les contours (par exemple toujours aller au plus proche) raccourcit le travail de la machine.

</div>

# Appendices

## Appendice A. Aide-Mémoire
```
MODELE POUR TRACEUR
  ; Initialisation
  G21 G90            1. initialisation : millimètres, positions absolues
  G0 Z5              2. lever le stylo
  ; Dessin (répéter le bloc autant de fois par formes séparées)
  G0 X.. Y..         3. aller au point de départ (stylo levé : on ne dessine pas)
  G1 Z0 F..          4. poser le stylo (et régler la vitesse)
  G1 X.. Y..         5. tracer : autant de lignes G1 que nécessaire
  G0 Z5              6. lever le stylo
  ; Fin
  M2                 7. fin du programme

COMMANDES
  G0   déplacement rapide, sans travailler
  G1   ligne droite, en travaillant
  G2   arc horaire ↻        G2 X.. Y.. I.. J..
  G3   arc anti-horaire ↺   G3 X.. Y.. I.. J..
  G90  positions absolues   G91  déplacements relatifs
  G21  millimètres          G20  pouces
  M2   fin du programme     ;    commentaire

ARCS
  I = X du centre − X du départ
  J = Y du centre − Y du départ
  cercle complet : arrivée = départ

RAPPELS
  - la machine a une MÉMOIRE (position, stylo, mode, vitesse)
  - un trait part TOUJOURS de la position actuelle
  - prédire avant d'exécuter ; tester souvent, par petits morceaux
```

## Appendice B. Débogage
| Symptôme | Cause probable | Solution |
|---|---|---|
| Message rouge « Il reste un trou » | un `___` ou un `?` n'a pas été remplacé | remplacez-le par un nombre |
| Message « Utilisez le POINT » | virgule décimale (`12,5`) | écrivez `12.5` |
| Message « Je ne comprends pas… » | une lettre oubliée devant un nombre, ou une lettre inconnue | relisez la ligne mot par mot |
| Trait parasite **depuis l'origine** | le stylo a été posé sans être d'abord amené au bon endroit | `G0 X.. Y..` **avant** `G1 Z0` |
| Trait parasite **entre deux formes** | le stylo n'a pas été levé | `G0 Z5` entre les deux formes |
| **Rien ne s'affiche** | le stylo n'est jamais posé (`Z0` oublié), ou on est resté en relatif | `explain` : colonnes « Stylo » et « Mode » |
| Dessin **25 fois trop grand** | `G20` (pouces) au lieu de `G21` (mm) | `G21` en première ligne |
| Dessin **déformé** | une coordonnée est fausse (`Y6` au lieu de `Y60`) | comparez avec l'objectif, vérifiez les numéros de ligne |
| « Arc incohérent » | `I`/`J` ou arrivée faux : départ et arrivée pas à la même distance du centre | `I = Xc − Xd`, `J = Yc − Yd` |
| Le dessin **sort de la feuille** | coordonnées supérieures à 297 (X) ou 210 (Y) | vérifiez les nombres et les unités |